# Graph Neural Network Approach FDiGNN - Data Pre-Processing and Pre-Treatment

# Dataset: Malaria Dataset (MD)

This notebook has the initial reading of the data from its available sources, pre-processing and pre-treatment.

**Pre-processing and pre-treatment of the data is adapted from an in-house developed Metabolomics Data Analysis Software.**

## MD

### To obtain the required file (for the notebook to be runnable), go to https://www.metabolomicsworkbench.org/data/DRCCStudySummary.php?Mode=SetupRawDataDownload&StudyID=ST000578 and click `ST000578_AN000888_Results.txt`. Place the downloaded file in the same folder as this file.

LC-MS data obtained in Positive Ionization Mode. Samples are (almost all) triplicates of 273 human blood samples collected from 220 human subjects. A total of 840 samples were analysed.

- 237 samples infected with Malaria which had previously been infected without information on Chloroquine Resistance - used for analysis as **'P.Vivax+Prior'**
- 213 samples infected with Malaria which had not previously been infected without information on Chloroquine Resistance - used for analysis as **'P.Vivax+NoPrior'**
- 138 samples not infected with Malaria with no information on previous infections and on Chloroquine Resistance - 117 used for analysis as **'Control'**
- 99 samples infected with Malaria with no information on previous infections and susceptible to Chloroquine
- 45 samples not infected with Malaria which had not previously been infected without information on Chloroquine Resistance - used for analysis as **'Control'**
- 45 samples infected with Malaria which had previously been infected and were resistant to Chloroquine
- 45 samples infected with Malaria with no information on previous infections and resistant to Chloroquine
- 15 samples not infected with Malaria which had previously been infected without information on Chloroquine Resistance - used for analysis as **'Control'**
- 3 samples infected with Malaria which had not previously been infected and resistant to Chloroquine

### Notebook Organization:

Data Pre-Processing and Pre-Treatment Pipeline

- Reading Data
- Data Annotation and Formula Assignment
- Data-preprocessing (De-Duplication and Filtering)
- Data pre-treatment.
- Unsupervised statistical analysis of the dataset.

# Table of Contents <a class="anchor" id="toc"></a>

- **[Step 1: Upload your data](#step-1)**
  - **[Step 1.1: Define your groups and see data characterization](#step-1_1)**
  - **[Step 1.2: Annotate with Database(s)](#step-1_2)**
  - **[Step 1.3: Formula Assignment](#step-1_3)**
  - **[Step 1.4: De-duplicating annotations](#step-1_4)**
- **[Step 2: Basic processing and pre-treatment](#step-2)**
- **[Step 3: Find Common and Exclusive metabolites between the groups](#step-3)**
- **[Step 4: Unsupervised Statistical Analysis (PCA and HCA)](#step-4)**
- **[Step 5: Find Specific Compounds](#step-6)**

### Some common needed imports

In [ ]:
import pandas as pd
import numpy as np

import re
import math
from fractions import Fraction

from tqdm import tqdm

import itertools

import scipy.spatial.distance as dist
import scipy.cluster.hierarchy as hier
import scipy.stats as stats

import sklearn.model_selection

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib import ticker
from matplotlib import cm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import TwoSlopeNorm

import seaborn as sns
import pickle
import json

# Our Python package
import metabolinks as mtl
import metabolinks.transformations as transf

# metanalysis_standard.py file  (has to be in the same folder)
import metanalysis_standard as metsta
# MDiN_functions.py file  (has to be in the same folder)
import MDiN_functions as md
# form_assign_func.py file (has to be in the same folder)
import form_assign_func as form_afunc
# elips.py file (has to be in the same folder)
from elips import plot_confidence_ellipse

# Step 1: Upload your data <a class="anchor" id="step-1"></a>

**Back to [Table of Contents](#toc)**

### Data Matrix (Aligned Samples) Reading Section

What does this do?
- Produces a pandas DataFrame with your spectral data
- Replaces 0 values with numpy null objects (nan). This is necessary to know how many metabolites each sample has and to allow further data processing.

**Do not forget to obtain the file from Metabolomics Workbench. To obtain it, go to https://www.metabolomicsworkbench.org/data/DRCCStudySummary.php?Mode=SetupRawDataDownload&StudyID=ST000578 and click `ST000578_AN000888_Results.txt`. Place the downloaded file in the same folder as this file.**

In [ ]:
# Read the file with the data
filename = 'ST000578_AN000888_Results.txt' # Name of your file
file = pd.read_csv(filename, sep='\t')
file = file.set_index('Samples')
file = file.replace({0:np.nan})
#file = file.loc[~file.index.duplicated(keep='first')]

labels = list(file.loc['Factors'])
file = file.iloc[1:]
file = file.astype(float).replace({0:np.nan}) # Passing the values from strings to floats.
file

In [ ]:
## See type of labels
pd.Series(labels).value_counts()

In [ ]:
np.random.seed(587402)
# Select 3 of the 24 replicates of the sample 853678 by removing 21 of its entries
select853678 = np.random.choice([i for i in range(len(file.columns)) if file.columns[i].startswith('853678')], 21, replace=False)
new_labels = [labels[i] for i in range(len(labels)) if i not in select853678]
labels = new_labels
file = file.iloc[:, [i for i in range(len(file.columns)) if i not in select853678]]

Select only the desired samples with the correct classes from the datasets and build the target in function of it

In [ ]:
selection = []
new_labels = []
for i in labels:
    if i == 'Current Malaria Infection:P.Vivax | Prior Malaria Infection:YES | Chloroquine Resistance:N/A':
        selection.append(True)
        new_labels.append('P.Vivax+Prior')
    elif i == 'Current Malaria Infection:P.Vivax | Prior Malaria Infection:NO | Chloroquine Resistance:N/A':
        selection.append(True)
        new_labels.append('P.Vivax+NoPrior')
    elif i.startswith('Current Malaria Infection:None'):
        selection.append(True)
        new_labels.append('Control')
    else:
        selection.append(False)

file = file.loc[:, selection]
pd.Series(new_labels).value_counts()

### Identifying and Creating a column with mass values (for Data Annotation downstream)

In [ ]:
# m/z were obtained in Positive mode
idx_masses = 'Positive'

file.insert(0, 'Probable m/z', [float(i.split('_')[0]) for i in file.index])

file.index.name = 'Bucket label'

# Select the column with the masses to compare
if idx_masses == 'Neutral':
    mass_val_col = 'Neutral Mass'
else:
    mass_val_col = 'Probable m/z'

# Replaces zeros with numpy nans. Essential for data processing
file = file.replace({0.0:np.nan})
file

### Identifying Metadata and Sample columns

**Modify based on your data**

In [ ]:
# Identify metadata columns
prev_annotations_cols = [] # Select columns which contain compound annotations
prev_formula_cols = [] # Select columns which contain formula assignments

other_metadata_columns = [] # Select other metadata columns in the dataset which will not be used for any specific
# analysis but are not sample columns

mass_val_col = mass_val_col # If instead of the mass column created, you have another column with masses you prefer to use
# change this mass_val_col to the na
    
# Identify QC samples
qc_sample_cols = [i for i in file.columns if i.startswith('SP')]

##########

metadata_cols = prev_annotations_cols + prev_formula_cols + [mass_val_col,] + other_metadata_columns

# Identify sample columns - automatic based on metadata columns selected
sample_cols = []
for c in file.columns:
    if c not in metadata_cols:
        if c not in qc_sample_cols:
            sample_cols.append(c)
#print(sample_cols)

# Step 1.1: Define your groups and see data characterization <a class="anchor" id="step-1_1"></a>

**Back to [Table of Contents](#toc)**

Here, we will define our sample classes, so that we can obtain merged dataframes for each one of them and define colours to help us visualise them in later figures.

In [ ]:
target = new_labels

# See if the classes are those that you want
classes = list(pd.unique(pd.Series(target)))
classes

In [ ]:
pd.Series(target).value_counts()

#### Colors for plots to ensure consistency

In [ ]:
# customize label colors

colours = sns.color_palette('tab10', 10) # Only room for 10 classes in this case, choose your colours
#colours = ('coral', 'turquoise', 'gold', 'indigo', 'lightgreen') # Example for using named colours
ordered_labels = classes # Put the classes, you can choose the order

label_colours = {lbl: c for lbl, c in zip(ordered_labels, colours)}
sample_colours = [label_colours[lbl] for lbl in target]

# See the colours for each class
sns.palplot(label_colours.values())
new_ticks = plt.xticks(range(len(ordered_labels)), ordered_labels)

### Initial Data Filtering.

In [ ]:
# Perform Data Filtering
filt_file = metsta.filtering_data_metabolomics(file, sample_cols, # DataFrame, Columns of Samples
                                   qc_sample_cols, target, # Columns of QC samples and the target

                               # Filter 1: Filter based on the number of samples features appear in
                               basic_filt='total_samples', # 'total_samples', 'class_samples', None
                               n_min_samples_feature_appear=30, # Min. number of samples a feature must appear in data/class

                               # Filter 2: Intensity based filter
                               int_based_filter=False, # True or False whether you apply it
                               intensity_calculation='mean', # 'median'
                               threshold_type='Intensity value', # '% Based'
                               threshold_value=5*10**5, # Fraction such as 0.1 for % Based

                               # Filter 3: QC sample feature variation based filter
                               QC_filter=False, # True or False whether you apply it
                               rsd_threshold=0.4, # Relative std threshold to remove features above said threshold

                               # Filter 4: Sample feature variance based filter
                               var_based_filter=True, # True or False whether you apply it
                               variance_calculation_type='Relative Standard Deviation', # 'Inter-Quartile Range',
                                #'Standard Deviation', 'Relative Standard Deviation', 'Median Absolute Deviation',
                                #'Relative Median Absolute Deviation'
                               feat_to_remove_percent=0.10, # Fraction of features to remove in this check

                               # Features to keep independent of filtering
                               feats_to_keep=[]
                               )
filt_file

In [ ]:
# See data characterization
data_characteristics = metsta.characterize_data(file[sample_cols].T, target=target)
data_characteristics

**Remove the reference feature if you have already normalized your data by the reference feature previously**

In [ ]:
ref_feat_lbl = ''

if ref_feat_lbl != '':
    filt_file = filt_file.drop(index=ref_feat_lbl)

# Step 1.2: Annotate with Database(s) <a class="anchor" id="step-1_2"></a>

**Back to [Table of Contents](#toc)**

Upload of HMDB dataset to annotate.

For the sake of simplicity, the lists of compound IDs, names and formulas will be placed in different columns. Just remember that they will be added by the same order, so the first ID in the IDs column corresponds to the first name in the names column and the first formula in the formulas column.

In [ ]:
dbs = { # How you want the database to be called in the data: 'HMDB', ' PCY', 'DBK'
    'HMDB': {'File': 'hmdb_complete.xlsx', # The name of each file
             'Index_name': 'accession', # The name of the index in each database
             'Name_col': 'name', # The name column in each database
             'Mass_col': None, # The mass column in each database. Can be None
             'Formula_col': 'chemical_formula'}, }

In [ ]:
for d in dbs:
    print(d, ' -> ', dbs[d]['File'], '|', dbs[d]['Index_name'],'|', dbs[d]['Mass_col'],'|', 
          dbs[d]['Name_col'],'|', dbs[d]['Formula_col'])

In [ ]:
# Upload databases
for d in dbs:
    print('Processing '+d)
    if dbs[d]['File'].endswith('.csv'):
        db = pd.read_csv(dbs[d]['File']).set_index(dbs[d]['Index_name'])
        if d == 'HMDB':
            db['name'] = db['name'].str.replace("b'", "")
            db['name'] = db['name'].str.replace("'", "")
    elif dbs[d]['File'].endswith('.xlsx'):
        db = pd.read_excel(dbs[d]['File']).set_index(dbs[d]['Index_name'])
        if d == 'HMDB':
            db['name'] = db['name'].str.replace("b'", "")
            db['name'] = db['name'].str.replace("'", "")
    else:
        raise ValueError('File Format not accepted. Only csv and xlsx files are accepted.')
    ##
    if dbs[d]['Mass_col'] == None:
        db['Calculated Mass'] = db[dbs[d]['Formula_col']].dropna().apply(metsta.calculate_monoisotopic_mass)
        dbs[d]['Mass_col'] = 'Calculated Mass'

    dbs[d]['DB'] = db
    print(d,'->', len(db.index), 'compounds')

#### Select which adducts you want to search for in the analysis

- Select adduct name and adduct mass shift in the dictionary - **At least one has to be selected to perform data annotation.**

This will calculate a mass based on the shift provided for each compound in the selected databases, which will then be used to search for and match to the _m/z_ or Neutral masses in your dataset.

In [ ]:
electron_mass = Fraction(0.000548579909065)

if idx_masses == 'Neutral':
    adducts_to_consider = {
        'Neutral': 0}
elif idx_masses == 'Positive':
    adducts_to_consider = {
        # Some common positive adducts to consider
        'H+': Fraction(metsta.chemdict['H']) - electron_mass,
        'Na+': Fraction(metsta.chemdict['Na']) - electron_mass,
        'K+': Fraction(metsta.chemdict['K']) - electron_mass,
    }
elif idx_masses == 'Negative':
    adducts_to_consider = {
        # Some common negative adducts to consider - Confirm if these are correct
        'H-': Fraction(- metsta.chemdict['H']) + electron_mass,
        'Cl-': Fraction(metsta.chemdict['Cl']) + electron_mass,
    }
else:
    adducts_to_consider = {} # No annotation is possible

In [ ]:
ppm_margin = 5
only_select_min_ppm = True # If True, when there are candidates for annotation with different ppm deviations, return only
# The ones with the lowest ppm deviation

# Annotation
annotated_data = filt_file.copy()
    
metsta.metabolite_annotation(annotated_data, dbs, ppm_margin, mass_val_col,
                             adducts_to_consider=adducts_to_consider, only_select_min_ppm=only_select_min_ppm)

# Step 1.3: Formula Assignment <a class="anchor" id="step-1_3"></a>

**Back to [Table of Contents](#toc)**

Skip for MD

In [ ]:
# Pass the formula assignment to your dataset
form_col = 'Formula_Assignment' # Name of the Column that will have the formula assignments

annotated_data[form_col] = ''

Roundup of Annotation and Formula Assignment Section Columns

In [ ]:
meta_cols = [i for i in annotated_data.columns if i not in sample_cols]
meta_cols_ids = [i for i in meta_cols if 'IDs' in i]
meta_cols_names = [i for i in meta_cols if 'names' in i]
meta_cols_formulas = [i for i in meta_cols if 'formulas' in i]
meta_cols_mcounts = [i for i in meta_cols if 'match count' in i]
print(meta_cols)
print('------------')
print(meta_cols_ids)
print('------------')
print(meta_cols_names)
print('------------')
print(meta_cols_formulas)
print('------------')
print(meta_cols_mcounts)

In [ ]:
# Comparing Formula Assignment with annotations
form_assigned_ann = 0
form_assigned_not_ann = 0
no_form_assigned = 0
idxs = []

if len(dbs) != 0:
    for idx in annotated_data.index:
        forms = []
        for col in meta_cols_formulas:
            form = annotated_data.loc[idx, col]
            if type(form) == list:
                forms.extend(annotated_data.loc[idx, col])
        forms = list(set(forms))
        #print(forms)
        if len(forms) > 0:
            assigned_form = annotated_data.loc[idx, form_col]
            if type(assigned_form) == str:
                if assigned_form in forms:
                    form_assigned_ann += 1
                    idxs.append(idx)
                else:
                    form_assigned_not_ann += 1
            else:
                no_form_assigned += 1

    print('Formula Assignment is a formula that was annotated:', form_assigned_ann)
    print('Formula Assignment is not a formula that was annotated:', form_assigned_not_ann)
    print('No Formula was assigned when there was annotations:', no_form_assigned)

else:
    print('No Data Annotation was performed')

# Step 1.4: De-duplicating annotations <a class="anchor" id="step-1_4"></a>

**Back to [Table of Contents](#toc)**

Due to the proximity of some _m/z_ peaks, they can have the same exact compound annotation or formula.

The following part merges peaks that have the same compound annotation into one single peak. This should happen since a compound should not be split into multiple peaks generally. Usually there is one peak that is the 'main' one with much higher intensities across the samples, although some cases this does not happen with the two _m/z_ peaks have or the annotation comes from two different adducts.

**This should not be used when multiple databases are used.**

In general, our procedure is the following.

1) See peaks that have the same metabolite annotation by other databases.

2) See if the other compound annotations do not have different annotations for those peaks.

3) If not, save the meta data of the compound and formula annotations by the different databases.

4) Then create the new peak. Intensities kept are the highest intensities when comparing peaks from the same adducts and summed when comparing peaks with annotations from different adducts. If peaks come from different adducts, meta data columns become identical to the peak which has the highest average intensity of all the peaks with the same annotation. If the peaks come all from the same adduct, then, if one peak has the highest intensity for all samples, then that peak becomes the 'de facto' peak and all others are erased; else metadata becomes the weighted average (based on the average intensity of the peaks) of all the peaks with the same annotation.

5) This process is repeated for annotations first and then formulas. **Usually the number of de-duplications made by each database should decrease since when you de-duplciate duplicate assignments by one database, you are usually de-duplicating in others.**

### MD is an LC-MS Dataset, therefore the following changes were made:

**Note: When comparing peaks from the same adduct, intensities are also summed like they are from different adducts and not only the highest is kept. This is due to them possibly corresponding to different peaks separated due to liquid chromatography. Below is the altered version of the de-duplication function.**

In [ ]:
def duplicate_disambiguator(annotated_data, sample_cols, mcid, mass_col, prev_an_form_cols, multiple_adds=True, verbose=False):
    """Attempts to remove duplicate (or more) annotations of peaks by merging them when possible.

       See explanation of procedure in explanation of Step 1.3.

       returns: annotated_data (pandas DataFrame with data after merging), 
                mergings_performed (dictionary with summary of mergings made),
                merging_situations (dictionary with counts of times each situation of merging was used),
                merge_description (dictionary with descriptions of mergings made),
                merge_problems (dictionary with descriptions of possible problems to merge)
    """

    # To store results
    merge_problems = {}
    merging_situations = {'Overwrite': 0, 'Merge same adducts':0, 'Merge different adducts':0, 'Problems': 0}
    mergings_performed = dict(zip(mcid, [0,]*len(mcid)))
    merge_description = {}
    n_merged_peaks = 0
    n_dropped_peaks = 0
    n_total_peaks = len(annotated_data.index)
    #merge_number = 0
    old_n_merged_peaks = -1
    n_loops = 1

    # Make iterations while the difference between the new current problems and the old_problems is not 0
    # That is when they stop being resolved
    while n_merged_peaks - old_n_merged_peaks > 0:
        old_n_merged_peaks = n_merged_peaks
        merge_problems = {}
        n_problems = 0
        # For the different databases used
        for col in mcid:
            new_idxs = {} # To save the idx where the merged peaks will be
            lost_idxs = [] # To save the idxs which will have to be removed
            # Adjust names if necessary
            col_id = col
            if col not in prev_an_form_cols:
                col = 'Matched '+col+' IDs'
            # See repeating annotations
            repeating_names = annotated_data[col].value_counts()[annotated_data[col].value_counts()>1].index

            for name in repeating_names: # For each repeating annotation
                # Grab the part of the dataframe with the repeats
                if col in prev_an_form_cols:
                    subset_df = annotated_data[annotated_data[col] == name]
                else:
                    subset_df = annotated_data.loc[[i for i in annotated_data[col].index if annotated_data.loc[i, col] == name]]

                # See if merging is possible and storing meta data
                merge=True
                saving_annotations = {} # To store the annotations to keep in the merged line
                for col_alt in mcid: # All other databases
                    col_alt_id = col_alt
                    if col_alt not in prev_an_form_cols:
                        col_alt = 'Matched '+col_alt+' IDs'
                    if col_alt == col:
                        saving_annotations[col_alt_id] = subset_df[col_alt].value_counts().index[0]
                    if col_alt != col:
                        a = []
                        # See if there are annotations in the other databases
                        subset_notnull = subset_df[col_alt][subset_df[col_alt].notnull()]
                        if len(subset_notnull) < 1:
                            #if len(subset_notnull) == 1: # If there is only one, save it
                            #    saving_annotations[col_alt_id] = subset_notnull.value_counts().index[0]
                            continue
                            # If there is not, nothing to save
                        else:
                            n_annotations = len(subset_notnull.value_counts())
                            # If there is more than one but they are all the same, save it
                            if n_annotations == 1 and len(subset_notnull) == len(subset_df):
                                saving_annotations[col_alt_id] = subset_notnull.value_counts().index[0]
                                #print(subset_notnull.value_counts().index)
                                continue
                            # If there are multiple different annotations, PROBLEM
                            # Merging will not happen
                            else:
                                prob_idx = tuple(subset_df.index)
                                if prob_idx not in merge_problems.keys():
                                    merge_problems[prob_idx] = {'Nº of peaks':len(subset_df),
                                                                'Annotation': name,
                                                                'Col Id.': col_id,
                                                                'Poss. Reason': col_alt_id}
                                    n_problems += 1
                                merge=False

                # Starting the merging process if possible
                if merge:
                    n_masses = subset_df[mass_col] # Get the masses to match

                    # Get the idxs positions of the repeated annotations in the dataframe
                    idxs = [annotated_data.index.get_loc(subset_df.index[i]) for i in range(len(subset_df))]

                    min_idx = min(idxs) # Grab the minimum that will become the merged peak

                    idxs.remove(min_idx)
                    lost_idxs.extend(idxs) # Add the other to lost_idxs that will be removed when all this is over

                    overwrite = False #
                    adducts_summed = False

                    # Get the intensity values for the new merged line
                    if 'Matched '+col_id+' adducts' not in subset_df.columns: # If not made in our software, it can't see
                        if col_id+' Adduct' not in subset_df.columns:
                            new_line = subset_df[sample_cols].sum().replace({0:np.nan})
                        else:
                            add_series = subset_df[col_id+' Adduct'].value_counts().index
                            if len(add_series) == 1:
                                new_line = subset_df[sample_cols].sum().replace({0:np.nan})
                            else:
                                new_line = subset_df[sample_cols].copy()
                                dif_ads = []
                                for ad in add_series:
                                    idxs = [i for i in subset_df.index if subset_df.loc[i, col_id+' Adduct'] == ad]
                                    new_df = new_line.loc[idxs].sum()
                                    dif_ads.append(new_df)
                                new_line = pd.concat(dif_ads, axis=1).sum(axis=1).replace({0:np.nan})
                                adducts_summed = True
                    else:
                        add_series = subset_df['Matched '+col_id+' adducts'].value_counts().index
                        if len(add_series) == 1:
                            #print(subset_df[sample_cols])
                            new_line = subset_df[sample_cols].sum().replace({0:np.nan})
                            #print(new_line)
                            #print('-----')
                        else:
                            new_line = subset_df[sample_cols].copy()
                            dif_ads = []
                            for ad in add_series:
                                idxs = [i for i in subset_df.index if subset_df.loc[i, 'Matched '+col_id+' adducts'] == ad]
                                new_df = new_line.loc[idxs].sum()
                                dif_ads.append(new_df)
                            new_line = pd.concat(dif_ads, axis=1).sum(axis=1).replace({0:np.nan})
                            adducts_summed = True
                    # For each annotation, see if the highest intensity values all come from one line
                    # Probably a better way to do this
                    for i in range(len(subset_df)):
                        if new_line.notnull().sum() - (new_line == subset_df[sample_cols].iloc[i]).sum() == 0:
                            # If yes, then situation 1: Overwrite is the correct option here
                            overwrite = True

                            # The new id (bucket label) will be the same as in the line with all the higher intensities
                            # Store it
                            keep_id = subset_df.index[i]
                            new_idxs[annotated_data.index[min_idx]] = keep_id

                            # The new line will be identical to that line in terms of intensity
                            temp_full_new_line = subset_df.iloc[i].copy()

                            # Filling the meta data of the new line with the data stored in saving annotations
                            idx_max = subset_df.loc[:,sample_cols].mean(axis=1).idxmax()
                            for key in saving_annotations:
                                #col_alt = key
                                if key in prev_an_form_cols:
                                    #subset_notnull = subset_df[subset_df[col_alt].notnull()]
                                    #idx_max = subset_notnull.loc[:,sample_cols].mean(axis=1).idxmax()
                                    #temp_full_new_line.loc[key] = saving_annotations[key]
                                    # Column is our Formula Assignment column
                                    if key + ' Adduct' in annotated_data.columns:
                                        #temp = annotated_data.loc[[
                                        #    i for i in annotated_data.index if annotated_data.loc[i, key] == saving_annotations[key]]]
                                        #temp_full_new_line[key + ' Adduct'] = subset_df.iloc[0][key + ' Adduct']
                                        temp_full_new_line[[key, key+' Adduct']] = subset_df.loc[idx_max, [key, key+' Adduct']]
                                    else:
                                        temp_full_new_line.loc[key] = subset_df.loc[idx_max, key]

                                else:
                                    #col_alt = 'Matched '+col_alt+' IDs'
                                    #subset_notnull = subset_df[subset_df[col_alt].notnull()]
                                    #idx_max = subset_notnull.loc[:,sample_cols].mean(axis=1).idxmax()
                                    # Grab the rest of the columns with meta_data: IDs, names, formulas and match count
                                    #temp = annotated_data.loc[[
                                    #    i for i in annotated_data.index if annotated_data.loc[i,
                                    #                'Matched '+key+' IDs'] == saving_annotations[key]]]

                                    rel_cols = ['Matched '+key+' IDs', 'Matched '+key+' names', 'Matched '+key+' formulas',
                                                'Matched '+key+' adducts', key+' match count']
                                    if key == 'HMDB':
                                        if 'Matched KEGG IDs' in annotated_data.columns:
                                            rel_cols.append('Matched KEGG IDs')
                                    #temp_full_new_line[rel_cols] = temp.iloc[0][rel_cols]
                                    temp_full_new_line[rel_cols] = subset_df.loc[idx_max, rel_cols]

                            merging_situations['Overwrite'] = merging_situations['Overwrite'] + 1
                            mergings_performed[col_id] = mergings_performed[col_id] + 1
                            merge_description[keep_id] = {'DB': col_id, 'Repeating annotation': name,
                                    'Nº merged peaks': len(subset_df), 'Merged peaks': list(subset_df.index), 'Situation': 'Overwrite'}

                            # Putting the merged line in the DataFrame
                            annotated_data.loc[annotated_data.index[min_idx]] = temp_full_new_line.copy()

                            if verbose:
                                if adducts_summed:
                                    print('Adducts were summed:', list(subset_df.index), 'to', keep_id)

                                if col_id in prev_an_form_cols:
                                    print(f'Merging {len(subset_df)} peaks (bucket labels: {list(subset_df.index)}) into one ', end='')
                                    print(f'({subset_df.index[i]}) by overwriting due to repeating annotations by previous ', end='')
                                    print(f'Annotation/Formula {col_id}: {name}.')
                                    print('----------')
                                else:
                                    print(f'Merging {len(subset_df)} peaks (bucket labels: {list(subset_df.index)}) into one ', end='')
                                    print(f'({subset_df.index[i]}) by overwriting due to repeating annotations by {col_id}: {name}.')
                                    print('----------')
                            continue

                    # If Situation 1, we end here
                    if overwrite:
                        continue

                    # If not Situation 1
                    temp_full_new_line = annotated_data.iloc[min_idx].copy()
                    # The new line will have the higher intensities for each sample
                    temp_full_new_line.loc[sample_cols] = new_line

                    # Filling the meta data of the new line with the data stored in saving annotations
                    idx_max = subset_df.loc[:,sample_cols].mean(axis=1).idxmax()
                    for key in saving_annotations:
                        #col_alt = key
                        if key in prev_an_form_cols:
                            #subset_notnull = subset_df[subset_df[col_alt].notnull()]
                            #idx_max = subset_notnull.loc[:,sample_cols].mean(axis=1).idxmax()
                            #temp_full_new_line.loc[key] = saving_annotations[key]
                            # Column is our Formula Assignment column
                            if key + ' Adduct' in annotated_data.columns:
                                #temp = annotated_data.loc[[
                                #    i for i in annotated_data.index if annotated_data.loc[i, key] == saving_annotations[key]]]
                                #temp_full_new_line[key + ' Adduct'] = subset_df.iloc[0][key + ' Adduct']
                                temp_full_new_line[[key, key+' Adduct']] = subset_df.loc[idx_max, [key, key+' Adduct']]
                            else:
                                temp_full_new_line.loc[key] = subset_df.loc[idx_max, key]

                        else:
                            #col_alt = 'Matched '+col_alt+' IDs'
                            #subset_notnull = subset_df[subset_df[col_alt].notnull()]
                            #idx_max = subset_notnull.loc[:,sample_cols].mean(axis=1).idxmax()
                            # Grab the rest of the columns with meta_data: IDs, names, formulas and match count
                            #temp = annotated_data.loc[[
                            #    i for i in annotated_data.index if annotated_data.loc[i,
                            #                'Matched '+key+' IDs'] == saving_annotations[key]]]
                            rel_cols = ['Matched '+key+' IDs', 'Matched '+key+' names', 'Matched '+key+' formulas',
                                        'Matched '+key+' adducts', key+' match count']
                            if key == 'HMDB':
                                if 'Matched KEGG IDs' in annotated_data.columns:
                                    rel_cols.append('Matched KEGG IDs')
                            #temp_full_new_line[rel_cols] = temp.iloc[0][rel_cols]
                            temp_full_new_line[rel_cols] = subset_df.loc[idx_max, rel_cols]

                    # All that's left is the bucket label, Neutral Mass and m/z
                    if multiple_adds:
                        # See if the distance between the maximum and minimum mass is low aka they come from the same adduct
                        # If yes, Situation 2: bucket label, Neutral Mass and m/z peak will be the weighted averages of all the
                        # possible peaks
                        if max(n_masses) - min(n_masses) < 0.5:
                            # Get the new bucket label
                            new_bucket = np.average(n_masses, weights=subset_df[sample_cols].mean(axis=1))
                            new_idxs[annotated_data.index[min_idx]] = new_bucket
                            # Get the Neutral Mass
                            temp_full_new_line[mass_col] = new_bucket

                            # Storing info
                            situation = 'merging'
                            merging_situations['Merge same adducts'] = merging_situations['Merge same adducts'] + 1
                            mergings_performed[col_id] = mergings_performed[col_id] + 1
                            merge_description[new_bucket] = {'DB': col_id, 'Repeating annotation': name,
                                    'Nº merged peaks': len(subset_df), 'Merged peaks': list(subset_df.index),
                                    'Situation': 'Merging same adducts'}

                        else:
                            # Get the new bucket label
                            argmax_idx = subset_df.loc[:,sample_cols].mean(axis=1).argmax()
                            new_bucket = subset_df.iloc[argmax_idx].name
                            new_idxs[annotated_data.index[min_idx]] = new_bucket
                            temp_full_new_line[mass_col] = subset_df.iloc[argmax_idx][mass_col]

                            # Storing info
                            situation = 'merging different adducts'
                            merging_situations['Merge different adducts'] = merging_situations['Merge different adducts'] + 1
                            mergings_performed[col_id] = mergings_performed[col_id] + 1
                            merge_description[new_bucket] = {'DB': col_id, 'Repeating annotation': name,
                                    'Nº merged peaks': len(subset_df), 'Merged peaks': list(subset_df.index),
                                    'Situation': 'Merging different adducts'}


                    else:
                        # Get the new bucket label
                        new_bucket = np.average(n_masses, weights=subset_df[sample_cols].mean(axis=1))
                        new_idxs[annotated_data.index[min_idx]] = new_bucket
                        temp_full_new_line[mass_col] = np.average(n_masses, weights=subset_df.loc[:,sample_cols].mean(axis=1))

                        # Storing info
                        situation = 'merging'
                        merging_situations['Merge same adducts'] = merging_situations['Merge same adducts'] + 1
                        mergings_performed[col_id] = mergings_performed[col_id] + 1
                        merge_description[new_bucket] = {'DB': col_id, 'Repeating annotation': name, 
                                    'Nº merged peaks': len(subset_df), 'Merged peaks': list(subset_df.index),
                                    'Situation': 'Merging same adducts (no multiple ad.)'}

                    # Putting the merged line in the DataFrame
                    annotated_data.loc[annotated_data.index[min_idx]] = temp_full_new_line.copy()

                    if adducts_summed:
                        print('Adducts were summed:', list(subset_df.index), 'to', new_bucket)

                    if verbose:
                        if col_id in prev_an_form_cols:
                            print(f'Merging {len(subset_df)} peaks (bucket labels: {list(subset_df.index)}) into one ', end='')
                            print(f'({new_bucket}) by {situation} due to repeating annotations by previous ', end='')
                            print(f'Annotation/Formula {col_id}: {name}.')
                            print('----------')
                        else:
                            print(f'Merging {len(subset_df)} peaks (bucket labels: {list(subset_df.index)}) into one ', end='')
                            print(f'({new_bucket}) by {situation} due to repeating annotations by {col_id}: {name}.')
                            print('----------')

            # Removing lost idxs peaks
            #print(len(lost_idxs), len(set(lost_idxs)))
            named_idxs = []
            for idx in lost_idxs:
                named_idxs.append(annotated_data.index[idx])
            for idx in named_idxs:
                annotated_data = annotated_data.drop(index=idx)
                n_dropped_peaks +=1

            # Assigning the new bucket labels
            for old_idx, new_idx in new_idxs.items():
                annotated_data = annotated_data.rename(index= {old_idx : new_idx})
                n_merged_peaks += 1
        print(f'Finished Loop Nº {n_loops}. Performed {n_merged_peaks - old_n_merged_peaks} merges.')
        n_loops += 1

    merging_situations['Problems'] = n_problems
    print('-----------')
    print(f'Nº of Max. Problems in Merging found (True value might be lower): {n_problems}')

    print('-----------')
    print('Nº of Mergings:            ', n_merged_peaks)
    print('Nº of Peaks merged:        ', n_merged_peaks + n_dropped_peaks)
    print('Nº of Peaks dropped:       ', n_dropped_peaks)
    print('Nº of Peaks before merging:', n_total_peaks)
    print('Nº of Peaks after merging: ', len(annotated_data.index))

    return annotated_data, mergings_performed, merging_situations, merge_description, merge_problems

In [ ]:
if len(prev_annotations_cols) > 0:
    mcid = prev_annotations_cols + list(dbs.keys())
    
else:
    mcid = list(dbs.keys())

if form_col in annotated_data.columns:
    mcid.append(form_col)

if len(prev_formula_cols) > 0:
    mcid = mcid + prev_formula_cols
    
#if formula_assignment > 0:
#    mcid = mcid + prev_formula_cols

Duplicate (or more) annotations report

In [ ]:
prev_an_form_cols = prev_annotations_cols + prev_formula_cols

for col in mcid:
    n_duplicates = []
    if col == form_col:
        col_alt = form_col
    elif col not in prev_an_form_cols:
        col_alt = 'Matched '+col+' IDs'
    else:
        col_alt = col
        col = 'Prev. Annotation: ' + col
    for i in annotated_data[annotated_data[col_alt].notnull()][col_alt]:
        a = 0
        for j in annotated_data[annotated_data[col_alt].notnull()][col_alt]:
            if i==j:
                if a == 1:
                    #print(i)
                    n_duplicates.append(i)
                    break
                a+=1
    print(col)
    print('Nº of same annotations on multiple peaks:        ', len(n_duplicates))
    print('Total number of annotations for these cases:     ', len(pd.Series(n_duplicates, dtype='object').value_counts()))
    if len(pd.Series(n_duplicates, dtype='object').value_counts()) == 0:
        print('Maximum number of peaks with the same annotation:', 0)
    else:
        print('Maximum number of peaks with the same annotation:', pd.Series(n_duplicates).value_counts().iloc[0])
    print('---------')

In [ ]:
old_data = annotated_data.copy()

In [ ]:
perform_deduplication = True # False
verbose = False
multiple_adds = True if len(adducts_to_consider) > 1 else False
prev_an_form_cols = prev_annotations_cols + prev_formula_cols

if perform_deduplication:
    annotated_data,mergings_performed,merging_situations,merge_description,merge_problems = metsta.duplicate_disambiguator(
        annotated_data, # Our data
        sample_cols, # Columns where the samples are
        mcid=mcid,
        mass_col=mass_val_col,
        prev_an_form_cols=prev_an_form_cols + [form_col,], # Previous Annotation and Formula Columns and Formula Assignment
        multiple_adds=multiple_adds, # If you have an m/z column
        verbose=verbose) # If you want a more detailed output while the function runs
else:
    mergings_performed, merging_situations, merge_description, merge_problems = [], [], [], []

### Seeing problems in merging

In [ ]:
problem_df = pd.DataFrame(merge_problems)
#problem_df

### Description of merging process

In [ ]:
m_desc = pd.DataFrame(merge_description)
m_desc

In [ ]:
if len(m_desc)>0:
    print('Nº of Mergings:            ', len(m_desc.columns))
    print('Nº of Peaks merged:        ', m_desc.loc['Nº merged peaks'].sum())
    print('Nº of Peaks dropped:       ', m_desc.loc['Nº merged peaks'].sum() - len(m_desc.columns))
    print('Nº of Peaks after merging: ', len(annotated_data.index))

In [ ]:
mergings_performed

In [ ]:
merging_situations

In [ ]:
print('Checking all matches')

cols_to_see = []
for i in prev_annotations_cols:
    if i in annotated_data.columns:
        cols_to_see.append(i)

if len(dbs) != 0:
    cols_to_see = cols_to_see + meta_cols_ids
annotated_data['Has Match?'] = np.nan
for i in tqdm(annotated_data.index):
    df = annotated_data.loc[[i]]
    hasmatch = df[cols_to_see].notnull().values.any()
    annotated_data.at[i, 'Has Match?'] = hasmatch
print('Nº of Annotated Compounds:', (annotated_data['Has Match?'] == True).sum())
print('---------------')

annotated_data.index = annotated_data.index.astype(str)
metadata_cols.append('Has Match?')
annotated_data.info(verbose= True)
annotated_data

In [ ]:
anns = 0
for i in annotated_data.index:
    if annotated_data.loc[i, 'Has Match?']:
        anns += 1
    #elif type(annotated_data.loc[i, 'Formula_Assignment']) == str:
        #print(annotated_data.loc[i, 'Formula_Assignment'])
    #    anns += 1
print(f'Number of annotated features: {anns}')

# Step 2: Basic processing and pre-treatment <a class="anchor" id="step-2"></a>

**Back to [Table of Contents](#toc)**

These functions are compilations from the pre-treatments available in the **Metabolinks** Python package.

Each step of this process has a different associated function that explain different methods available to do those steps included in the `filtering_pretreatment` function. By our experience, the default option in the different functions are the most common ones to use.

This returns five DataFrames:
- **treated_data** - Data after filtering and pre-treatment with the samples ready for statistical analysis.
- **processed_data** - Data after filtering and only normalization with samples and meta data used for compound finding and distinguishing between common and exclusive metabolites.
- **univariate_data** - Data after filtering, imputation and only normalization used for fold change calculation in univariate analysis.
- **meta_data** - Meta data with compound annotation and formulas for later.
- **bin_data** - treated_data but with BinSim just because.

**The procedures to be used need to be chosen by the user.**

### Feature Filtering - `basic_feat_filtering` function

This part removes the features that appear only in one sample (likely experimental artifacts and not real metabolites). If this is already done, skip this part by turning **_filt_method argument_ in `filtering_pretreatment` function to None**.

**Available methods**: 'total_samples' (_default_), 'class_samples', None.

**There can also be an extra step just keeping masses with annotations**: 'Formula', 'Name', None (_default_). Explained in the function.

### Data Pre-Treatment

There are many different ways these can be used but in general there are four categories: 'Missing Value Imputation', 'Normalization', 'Transformations' and 'Scaling' each with their options. If you do not want some types of pre-treatment, select None for that specific category (except missing value imputation, that HAS to be done).

#### Note: If data was already normalized, skip normalization by making _norm_ argument in `filtering_pretreatment` function to None and remove reference feature if you have it.

Each different method for each category is explained in their respective functions. Each category has also a keyword (kw) that can be added since many methods have one parameter that can be changed. That keyword becomes that parameter.

**Missing Value Imputation** (`missing_value_imputer`): 'min_sample' (_Default_), 'min_feat', 'min_data', 'zero'.

**Normalization** by (`normalizer`): 'ref_feat' (_Default_), 'total_sum', 'PQN', 'Quantile', None.

**Transformation** (`transformer`): 'glog' (_Default_), None.

**Scaling** (`scaler`): 'pareto' (_Default_), 'mean_center', 'auto', 'range', 'vast', 'level', None.

Furthermore, **Binary Simplification** (BinSim) is also returned as well and kept in bin_data.

In [ ]:
# Filtering based on number of times features appear
filt_method='total_samples' # 'total_samples', 'class_samples', None
filt_kw=30 # Nº of minimum samples of the dataset ('total_samples') or class ('class_samples') features have to appear in
extra_filt=None # Filtering based on annotation of features 'Formula', 'Name' or None

# Missing Value Imputations
mvi='min_sample' # 'min_sample' (Default), 'min_feat', 'min_data', 'zero'
mvi_kw=1/5 # Specific Keyword for MVI method

# Normalization
norm='total_sum' # 'ref_feat' (Default), 'total_sum', 'PQN', 'Quantile', None
norm_kw='555.2692975341 Da' # Specific keyword for Normalization method

# Transformation
tf='glog' # 'glog' (Default), None
tf_kw=None # Specific keyword for Transformation

# Scaling
scaling='pareto' # 'pareto' (Default), 'mean_center', 'auto', 'range', 'vast', 'level', None
scaling_kw=None # Specific keyword for Scaling

# Change the parameters in the variables above
treated_data, processed_data, univariate_data, meta_data, bin_data = metsta.filtering_pretreatment(
                  annotated_data, target,sample_cols,
                  filt_method, filt_kw, extra_filt, # Filtering 
                  mvi, mvi_kw, # Missing value imputation
                  norm, norm_kw, # Normalization
                  tf, tf_kw, # Transformation
                  scaling, scaling_kw) # Scaling

In [ ]:
treated_data

In [ ]:
#processed_data.info()
processed_data

In [ ]:
# See data characterization
data_characteristics = metsta.characterize_data(processed_data[sample_cols].T, target=target)
data_characteristics

In [ ]:
print(f'Sample with the lowest number of metabolic features:  {bin_data.sum(axis=1).min()}')
print(f'Mean number of metabolic features per sample:         {bin_data.sum(axis=1).mean():.3f}')
print(f'Sample with the highest number of metabolic features: {bin_data.sum(axis=1).max()}')

In [ ]:
save_pretreated_data = True
# Filename for the exported data
filename_TreatedData = 'Data/MD_AllTreatedData_Final.xlsx'
# Filename for the exported data pickle
filename_proc = 'Data/MD_ProcData_Final.pickle'
filename_treat = 'Data/MD_TreatedData_Final.pickle'
# Filename for the exported target
target_name = 'Data/MD_Target_Final.txt'

if save_pretreated_data:
    # Saving data
    with pd.ExcelWriter(filename_TreatedData) as writer:
    #    processed_data.to_excel(writer, sheet_name='Metadata+Normalized Data')
    #    treated_data.T.to_excel(writer, sheet_name='Fully Treated Data')
        bin_data.T.to_excel(writer, sheet_name='BinSim Treated Data')
        univariate_data.T.to_excel(writer, sheet_name='MVI+Norm Data')
        annotated_data.to_excel(writer, sheet_name='Non Treated')

    # Saving data
    processed_data.to_pickle(filename_proc)
    treated_data.T.to_pickle(filename_treat)


    # Saving data
    with open(target_name, 'w') as f:
        f.write('\n'.join(target))

# Step 3: Find Common and Exclusive metabolites between the groups <a class="anchor" id="step-3"></a>

**Back to [Table of Contents](#toc)**

First, make a dataframe for each class with only the features that appear in those classes. We will also make another set of DataFrames only considering annotated metabolites.

In [ ]:
# Sample specific data frames
groups = {}
group_dfs = {}

group_dfs_ids = {}

for cl in classes:
    groups[cl] = []
    
for c, t in zip(processed_data[sample_cols].columns, target):
    for g in groups:
        if g == t:
            groups[g].append(c)
            
for g in groups:
    for c, t in zip(processed_data[sample_cols].columns, target):
        if g == t:
            group_dfs[g] = processed_data.dropna(subset= groups[g], thresh=1)
            group_dfs_ids[g] = group_dfs[g].iloc[[
                i for i in range(len(group_dfs[g]['Has Match?'])) if group_dfs[g]['Has Match?'].iloc[i]]]

for df in group_dfs:
    print(df,  '------>', len(group_dfs[df]), 'metabolites from which', len(group_dfs_ids[df]), f'have matches')

We can now see the common and exclusive metabolites between these DataFrames.

In [ ]:
# Common to all
common_all = metsta.common(group_dfs.values())
common_all_id = metsta.common(group_dfs_ids.values())
print(len(common_all.index), f'metabolites are common to all group, {len(common_all_id.index)} with matches.')

print('    ')

# Common to two or more - Might become unintelligible if you have too many classes
for n in range(2, len(group_dfs)+1):
    for comb in itertools.combinations(group_dfs, n):
        df_list = []
        labels_list = []
        df_id_list = []
        for c in comb:
            labels_list.append(c)
            df = group_dfs[c]
            df_list.append(df)
            df_id_list.append(group_dfs_ids[c])
        df_common = metsta.common(df_list)
        df_id_common = metsta.common(df_id_list)
        for s in group_dfs:
            if s not in labels_list:
                exclude = group_dfs[s]
                df_common = df_common.loc[~(df_common.index.isin(exclude.index))]
                df_id_common = df_id_common.loc[~(df_id_common.index.isin(group_dfs_ids[s].index))]
        print(len(df_common.index), f'metabolites ({len(df_id_common.index)} with matches) are common to {comb}.') 

print('     ')

# Exclusive to only one group
exclusives = metsta.exclusive(group_dfs.values())
exclusives_id = metsta.exclusive(group_dfs_ids.values())
exc_id = dict(zip(group_dfs, exclusives_id))
exc = dict(zip(group_dfs, exclusives))
for g in group_dfs:
    print(len(exc[g].index), f'metabolites are exclusive to {g}, {len(exc_id[g].index)} with matches.')

# Step 4: Unsupervised Statistical Analysis <a class="anchor" id="step-4"></a>

**Back to [Table of Contents](#toc)**

Unsupervised analysisincldue PCA and Hierarchical Clustering (HCA) Analysis.

In [ ]:
def plot_PCA(principaldf, label_colors, components=(1,2), title="PCA", ax=None):
    "Plot the projection of samples in the 2 main components of a PCA model."
    
    if ax is None:
        ax = plt.gca()
    
    loc_c1, loc_c2 = [c - 1 for c in components]
    col_c1_name, col_c2_name = principaldf.columns[[loc_c1, loc_c2]]

    #ax.axis('equal')
    ax.set_xlabel(f'{col_c1_name}')
    ax.set_ylabel(f'{col_c2_name}')

    unique_labels = principaldf['Label'].unique()

    for lbl in unique_labels:
        subset = principaldf[principaldf['Label']==lbl]
        ax.scatter(subset[col_c1_name],
                   subset[col_c2_name],
                   s=50, color=label_colors[lbl], label=lbl)

    #ax.legend(framealpha=1)
    ax.set_title(title, fontsize=15)

def plot_ellipses_PCA(principaldf, label_colors, components=(1,2),ax=None, q=None, nstd=2):
    "Plot confidence ellipses of a class' samples based on their projection in the 2 main components of a PCA model."
    
    if ax is None:
        ax = plt.gca()
    
    loc_c1, loc_c2 = [c - 1 for c in components]
    points = principaldf.iloc[:, [loc_c1, loc_c2]]
    
    #ax.axis('equal')

    unique_labels = principaldf['Label'].unique()

    for lbl in unique_labels:
        subset_points = points[principaldf['Label']==lbl]
        plot_confidence_ellipse(subset_points, q, nstd, ax=ax, ec=label_colors[lbl], fc='none')

def color_list_to_matrix_and_cmap(colors, ind, axis=0):
        if any(issubclass(type(x), list) for x in colors):
            all_colors = set(itertools.chain(*colors))
            n = len(colors)
            m = len(colors[0])
        else:
            all_colors = set(colors)
            n = 1
            m = len(colors)
            colors = [colors]
        color_to_value = dict((col, i) for i, col in enumerate(all_colors))

        matrix = np.array([color_to_value[c]
                           for color in colors for c in color])

        matrix = matrix.reshape((n, m))
        matrix = matrix[:, ind]
        if axis == 0:
            # row-side:
            matrix = matrix.T

        cmap = mpl.colors.ListedColormap(all_colors)
        return matrix, cmap

def plot_dendogram(Z, leaf_names, label_colors, title='', ax=None, no_labels=False, labelsize=12, **kwargs):
    if ax is None:
        ax = plt.gca()
    hier.dendrogram(Z, labels=leaf_names, leaf_font_size=10, above_threshold_color='0.2', orientation='left',
                    ax=ax, **kwargs)
    #Coloring labels
    #ax.set_ylabel('Distance (AU)')
    ax.set_xlabel('Distance (AU)')
    ax.set_title(title, fontsize = 15)
    
    #ax.tick_params(axis='x', which='major', pad=12)
    ax.tick_params(axis='y', which='major', labelsize=labelsize, pad=12)
    ax.spines['left'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    #xlbls = ax.get_xmajorticklabels()
    xlbls = ax.get_ymajorticklabels()
    rectimage = []
    for lbl in xlbls:
        lbl_text = lbl.get_text()
        if type(list(label_colors)[0]) == np.float64:
            lbl_text = float(lbl_text)
        col = label_colors[lbl_text]
        lbl.set_color(col)
        #lbl.set_fontweight('bold')
        if no_labels:
            lbl.set_color('w')
        rectimage.append(col)

    cols, cmap = color_list_to_matrix_and_cmap(rectimage, range(len(rectimage)), axis=0)

    axins = inset_axes(ax, width="5%", height="100%",
                   bbox_to_anchor=(1, 0, 1, 1),
                   bbox_transform=ax.transAxes, loc=3, borderpad=0)

    axins.pcolor(cols, cmap=cmap, edgecolors='w', linewidths=1)
    axins.axis('off')

### Principal Component Analysis (PCA)

**For Supplementary Figure 1C.**

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(6,6)) # Change the size of the figure

principaldf, var, loadings = metsta.compute_df_with_PCs_VE_loadings(treated_data, 
                                       n_components=2, # Select number of components to calculate
                                       whiten=True, labels=target, return_var_ratios_and_loadings=True)

# Plot PCA
ax.axis('equal')
lcolors = label_colours

plot_PCA(principaldf, lcolors, 
         components=(1,2), # Select components to see
         title='', # Select title of plot
         ax=ax)

# Remove ellipses by putting a # before the next line
plot_ellipses_PCA(principaldf, 
                  lcolors, 
                  components=(1,2), # Select components to see
                  ax=ax, 
                  q=0.95) # Confidence ellipse with 95% (q) confidence

ax.set_xlabel(f'PC 1 ({var[0] * 100:.1f} %)', size=15) # Set the size of labels
ax.set_ylabel(f'PC 2 ({var[1] * 100:.1f} %)', size=15) # Set the size of labels

plt.legend(fontsize=15) # Set the size of labels
#plt.grid() # If you want a grid or not
plt.show()
#f.savefig('Name_PCAplot.png', dpi=600) # Save the figure

### Hierarchical Clustering Analysis (HCA)

Performing Hierarchical Clustering.

Distance metrics: 'euclidean' is the default, others are in https://docs.scipy.org/doc/scipy/reference/spatial.distance.html.

Linkage metrics: **'ward', 'average'**, 'centroid', 'single', 'complete', 'weighted', 'median'.

In [ ]:
metric = 'euclidean' # Select distance metric
method = 'ward' # Select linkage method

distances = dist.pdist(treated_data, metric=metric)
Z = hier.linkage(distances, method=method)

hca_res = {'Z': Z, 'distances': distances}

In [ ]:
# Plot HCA
with sns.axes_style("white"):
    f, ax = plt.subplots(1, 1, figsize=(4, 4), constrained_layout=True) # Set Figure Size
    plot_dendogram(hca_res['Z'], 
                   target, ax=ax,
                   label_colors=label_colours,
                   title='', # Select title
                   color_threshold=0) # Select a distance threshold from where different sets of lines are coloured

    plt.show()
    #f.savefig('Name_HCAplot.png', dpi=400) # Save the figure

# Step 5: Find Specific Compounds <a class="anchor" id="step-6"></a>

**Back to [Table of Contents](#toc)**

This code allows to find specific compounds by their index, name, formula, or neutral mass / Probable _m/z_.

In [ ]:
id_type = 'index' #pick 'formula', 'name', 'Neutral Mass'/'Probable m/z', 'index', or 'kegg'
find_id = '130.0639_306.4'
if id_type == 'index':
    finder = processed_data.loc[processed_data.index.str.startswith(find_id)]
elif id_type in ['Neutral Mass', 'Probable m/z']:
    finder = processed_data.loc[processed_data[mass_val_col].str.startswith(find_id)]
elif id_type == 'formula':
    index_list = []
    for col in meta_cols_formulas:
        temp_df = processed_data[[col]].dropna()
        for t in temp_df.index:
            if find_id in temp_df[col][t]:
                index_list.append(t)
    
    for col in prev_formula_cols + [form_col, ]:
        if col in processed_data.columns:
            temp_df = processed_data[col].dropna()
            for t in temp_df.index:
                if find_id in temp_df[t]:
                    index_list.append(t)
    finder = processed_data.loc[processed_data.index.isin(index_list)]
elif id_type == 'name':
    index_list = []
    for col in meta_cols_names:
        temp_df = processed_data[[col]].dropna()
        for t in temp_df.index:
            if find_id in temp_df[col][t]:
                index_list.append(t)
    for col in prev_annotations_cols:
        if col in processed_data.columns:
            temp_df = processed_data[col].dropna()
            for t in temp_df.index:
                if find_id in temp_df[t]:
                    index_list.append(t)
    finder = processed_data.loc[processed_data.index.isin(index_list)]
elif id_type == 'kegg':
    index_list = []
    if 'HMDB' in dbs:
        temp_df = processed_data[['Matched KEGG IDs']].dropna()
        for t in temp_df.index:
            if find_id in temp_df['Matched KEGG IDs'][t]:
                index_list.append(t)
        finder = processed_data.loc[processed_data.index.isin(index_list)]
    else:
        print('No HMDB annotation found. Please annotate with HMDB to have access to KEGG ID information')

finder

In [ ]:
# Plot to see the intensities distribution (might become difficult to see with a lot of samples)
bar_plot_info = finder.replace({np.nan:0})
if len(bar_plot_info.index) == 1:
    fig, ax = plt.subplots(1,1, figsize=(16,6))
    x = np.arange(len(bar_plot_info.columns[-len(sample_cols):]))
    for comps in range(len(bar_plot_info.index)):
        ax.barh(x, np.array(bar_plot_info.iloc[comps, -len(sample_cols):]), color=sample_colours)
        ax.set_ylabel('Normalized Intensity', fontsize=15)
        ax.set_title(find_id, fontsize=15)
else:
    fig, axs = plt.subplots(1,len(bar_plot_info.index), figsize=(16,6))

    x = np.arange(len(bar_plot_info.columns[-len(sample_cols):]))
    for (comps, ax) in zip(range(len(bar_plot_info.index)), axs.ravel()):
        ax.barh(x, np.array(bar_plot_info.iloc[comps, -len(sample_cols):]), color=sample_colours)
        ax.set_yticks([])
        #ax.tick_params(labelleft=False)
        ax.set_ylabel('Normalized Intensity', fontsize=13)
        ax.set_title(find_id)
    ax = axs[0]
ax.set_yticks(x)
ax.set_yticklabels(bar_plot_info.columns[-len(sample_cols):], fontsize=12)
plt.show()

Search for the differences between groups

In [ ]:
# Calculate average intensities
gfinder = finder.copy()
for g in groups:
    gfinder[g+' Average'] = gfinder[gfinder.columns.intersection(groups[g])].mean(axis=1)
    gfinder[g+' std'] = gfinder[gfinder.columns.intersection(groups[g])].std(axis=1)
gfinder

In [ ]:
avg_cols = [col for col in gfinder.columns if 'Average' in col]
std_cols = [col for col in gfinder.columns if 'std' in col]
group_colours = [label_colours[lbl] for lbl in classes]
bar_plot_info = gfinder.replace({np.nan:0})
if len(bar_plot_info.index) == 1:
    fig, ax = plt.subplots(1,1, figsize=(2,3))
    x = np.arange(len(avg_cols))
    for comps in bar_plot_info.index:
        ax.bar(x, np.array(bar_plot_info.loc[comps, avg_cols]), color=group_colours, yerr=np.array(bar_plot_info.loc[comps, std_cols]), capsize=12)
        ax.set_ylabel('Normalized Intensity', fontsize=15)
        ax.set_title(find_id, fontsize=15)
        ax.set_xticks(x)
        ax.set_xticklabels(classes, fontsize=15)
else:
    fig, axs = plt.subplots(1,len(bar_plot_info.index), figsize=(16,6))

    x = np.arange(len(avg_cols))
    for (comps, ax) in zip(bar_plot_info.index, axs.ravel()):
        ax.bar(x, np.array(bar_plot_info.loc[comps, avg_cols]), color=group_colours, yerr=np.array(bar_plot_info.loc[comps, std_cols]), capsize=12)
        ax.set_yticks([])
        #ax.tick_params(labelleft=False)
        ax.set_ylabel('Normalized Intensity', fontsize=13)
        ax.set_title(find_id)
        ax.set_xticks(x)
        ax.set_xticklabels(groups)
    ax = axs[0]
plt.show()

In [ ]:
pd.Series(np.array(target)[finder.iloc[:,9:].notnull().values[0]]).value_counts()/pd.Series(np.array(target)).value_counts()